# 아로코 전체 상품 스크래핑

브랜드 전체 상품 목록에서 상품 ID, 상품명, 가격, 품절 여부, 상세 URL, 이미지 URL을 수집합니다. 상품 정보가 HTML에 포함되어 있어 Selenium 없이 `requests`와 `BeautifulSoup`를 사용합니다.

> 공개 상품만 천천히 요청합니다. 이용약관과 robots 정책을 확인하고 요청 간격을 지나치게 줄이지 마세요.

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
import urllib3
from bs4 import BeautifulSoup

BASE_URL = "https://arokor.com"
LIST_URL = f"{BASE_URL}/product/list.html"
CATEGORY_NO = 23  # 브랜드 전체 상품
MAX_PAGES = 50
REQUEST_DELAY = 1.0
VERIFY_SSL = False  # 현재 교육 환경의 인증서 체인 오류 때문에 비활성화
if not VERIFY_SSL:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 Chrome/140.0 Safari/537.36"),
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
    "Referer": f"{BASE_URL}/",
})

## 상품 카드 파싱

`anchorBoxId_상품번호`와 `.description`을 기준으로 값을 읽고 상대 주소를 절대 주소로 바꿉니다.

In [ ]:
def absolute_url(value):
    if not value:
        return None
    return "https:" + value if value.startswith("//") else urljoin(BASE_URL, value)


def parse_product_card(card, page):
    name_link = card.select_one(".description .name a")
    name_spans = name_link.select(":scope > span") if name_link else []
    image = card.select_one(".thumbnail .prdImg img")
    description = card.select_one(".description")
    id_match = re.search(r"anchorBoxId_(\d+)", card.get("id", ""))
    price_text = description.get("ec-data-price", "") if description else ""
    price_digits = re.sub(r"[^0-9]", "", price_text)

    return {
        "product_id": int(id_match.group(1)) if id_match else None,
        "name": name_spans[-1].get_text(" ", strip=True) if name_spans else None,
        "price": int(price_digits) if price_digits else None,
        "sold_out": card.select_one('img[alt*="품절"], .soldout_icon') is not None,
        "product_url": absolute_url(name_link.get("href")) if name_link else None,
        "image_url": absolute_url(image.get("src")) if image else None,
        "page": page,
    }


def fetch_product_page(page):
    response = session.get(
        LIST_URL, params={"cate_no": CATEGORY_NO, "page": page},
        timeout=20, verify=VERIFY_SSL,
    )
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "lxml")
    cards = soup.select('ul.prdList > li[id^="anchorBoxId_"]')
    return [parse_product_card(card, page) for card in cards]

## 전체 페이지 수집

상품 카드가 없는 페이지에서 멈추고 상품 ID 기준으로 중복을 제거합니다.

In [ ]:
products = []

for page in range(1, MAX_PAGES + 1):
    page_products = fetch_product_page(page)
    if not page_products:
        print(f"{page}페이지: 상품 없음 → 수집 종료")
        break
    products.extend(page_products)
    print(f"{page}페이지: {len(page_products)}개 (누적 {len(products)}개)")
    time.sleep(REQUEST_DELAY)
else:
    print(f"MAX_PAGES={MAX_PAGES}에 도달했습니다. 필요하면 늘리세요.")

df = (pd.DataFrame(products)
      .dropna(subset=["product_id", "name"])
      .drop_duplicates(subset="product_id", keep="first")
      .sort_values("product_id")
      .reset_index(drop=True))

print(f"최종 수집 상품: {len(df):,}개")
df.head()

## 검증 및 CSV 저장

누락·중복·품절 건수를 확인하고 Excel에서도 한글이 깨지지 않도록 UTF-8 BOM으로 저장합니다.

In [ ]:
display(df.isna().sum().rename("missing_count"))
print("중복 상품 ID:", df["product_id"].duplicated().sum())
print("품절 상품:", int(df["sold_out"].sum()))

data_dir = Path("../data") if Path.cwd().name == "chapt999" else Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
output_path = data_dir / "arokor_products.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path.resolve()}")

## 앱용 데이터 내보내기

`data/arokor_enrichment.json`에 향 노트와 어코드를 관리합니다. 전체 상품을 추천 후보로 내보내며 미확인 정보는 화면에 ‘확인 전’으로 표시합니다. 같은 이름의 용량별 상품은 `product_ids`로 묶고, 세트·엘릭서·다른 농도는 별도 유지합니다. 새 수집 상품도 자동으로 포함합니다. 향 정보를 조사한 뒤 출처와 함께 JSON을 보완하고 다시 실행하세요. 생성된 파일을 앱 폴더와 mobile-build 폴더에 복사하면 수집 결과를 갱신할 수 있습니다.

In [ ]:
import importlib.util

project_root = Path.cwd().parent if Path.cwd().name == "chapt999" else Path.cwd()
exporter_path = project_root / "chapt999/export_arokor.py"
spec = importlib.util.spec_from_file_location("export_arokor", exporter_path)
exporter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exporter)
catalog_path = project_root / "chapt999/app-integration/arokor-catalog.js"
catalog = exporter.export_catalog(catalog_path)
print(f"앱 데이터: 상품 {len(catalog['products'])}개, 추천 후보 {len(catalog['recommendations'])}개, 향 정보 확인 {catalog['verifiedCount']}개")
print(f"내보내기 완료: {catalog_path}")